# 01 — Carga de Datos y Preprocesamiento

Este notebook cubre la ingesta, inspección inicial y limpieza del dataset de Meetup Tennessee. El output es un conjunto de CSVs limpios en `data/processed/` que serán consumidos por todos los notebooks posteriores.

## Dataset: Red Social de Meetup (Tennessee)

Este conjunto de datos ofrece una visión detallada de las interacciones entre usuarios y grupos en **meetup.com**, una plataforma diseñada para organizar y asistir a eventos presenciales. Fue creado originalmente para la charla *"Principles of Network Analysis with NetworkX"*, presentada en conferencias como **PyNash** y **PyTennessee**.

La información está dividida en dos categorías principales:

#### Datos de Grafos (Aristas)

| Archivo | Descripción | Peso (Weight) |
| --- | --- | --- |
| `member-to-group-edges.csv` | Red bipartita entre miembros y grupos | Número de eventos asistidos |
| `group-edges.csv` | Conexiones entre grupos | Miembros compartidos entre grupos |
| `member-edges.csv` | Conexiones entre miembros | Grupos compartidos entre personas |
| `rsvps.csv` | Datos crudos de asistencia | Base para generar las redes anteriores |

#### Metadatos (Nodos)

| Archivo | Descripción |
| --- | --- |
| `meta-groups.csv` | Detalles de cada grupo (nombre, categoría). Índice: `group_id` |
| `meta-members.csv` | Detalles de los usuarios (nombre, ubicación). Índice: `member_id` |
| `meta-events.csv` | Detalles de los eventos (nombre, fecha/hora). Índice: `event_id` |

## 1. Ingesta desde Kaggle

El dataset se descarga directamente desde Kaggle (`stkbailey/nashville-meetup`) usando `kagglehub`. Para garantizar la reproducibilidad sin depender de conexión a internet en iteraciones posteriores, cada archivo descargado se copia a `data/raw/` y se carga simultáneamente en un **diccionario de DataFrames** llamado `dataframes`, donde la clave es el nombre del archivo sin extensión.

Durante la carga se elimina la columna `Unnamed: 0` cuando aparece — es un residuo habitual al exportar DataFrames desde pandas que solo contiene el índice numérico y no aporta información.

In [1]:
import os
import shutil
import pandas as pd
import kagglehub
import seaborn as sns
import matplotlib.pyplot as plt

c:\Users\marco\.conda\envs\TFM_grafos_V2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Aseguramos que la carpeta data/raw existe
raw_dir = "../data/raw"
os.makedirs(raw_dir, exist_ok=True)

# Descarga del dataset desde Kaggle
path_kaggle = kagglehub.dataset_download("stkbailey/nashville-meetup")
archivos = os.listdir(path_kaggle)

# Diccionario para guardar los DataFrames en memoria
dataframes = {}

print("--- Cargando Archivos y copiando a RAW ---")

for archivo in archivos:
    # 1. Copiar el CSV original a data/raw/
    src_file = os.path.join(path_kaggle, archivo)
    dst_file = os.path.join(raw_dir, archivo)
    shutil.copy2(src_file, dst_file)

    # 2. Cargar en pandas
    df = pd.read_csv(src_file)

    # 3. Eliminar columna de índice residual si existe
    if 'Unnamed: 0' in df.columns:
        df.drop('Unnamed: 0', axis=1, inplace=True)

    nombre_clave = archivo.replace(".csv", "")
    dataframes[nombre_clave] = df

    filas, columnas = df.shape
    print(f"✅ Cargado y guardado: {nombre_clave} ({filas} filas x {columnas} columnas)")
    print("-" * 50)

print("--- Ingesta Completa ---")
print(f"Diccionario de DataFrames creado con claves:\n{list(dataframes.keys())}")

--- Cargando Archivos y copiando a RAW ---
✅ Cargado y guardado: group-edges (6692 filas x 3 columnas)
--------------------------------------------------
✅ Cargado y guardado: member-edges (1176368 filas x 3 columnas)
--------------------------------------------------
✅ Cargado y guardado: member-to-group-edges (45583 filas x 3 columnas)
--------------------------------------------------
✅ Cargado y guardado: meta-events (19307 filas x 4 columnas)
--------------------------------------------------
✅ Cargado y guardado: meta-groups (602 filas x 7 columnas)
--------------------------------------------------
✅ Cargado y guardado: meta-members (24591 filas x 7 columnas)
--------------------------------------------------
✅ Cargado y guardado: rsvps (126813 filas x 3 columnas)
--------------------------------------------------
--- Ingesta Completa ---
Diccionario de DataFrames creado con claves:
['group-edges', 'member-edges', 'member-to-group-edges', 'meta-events', 'meta-groups', 'meta-memb

## 2. Inspección Inicial

Antes de entrar en el preprocesamiento, inspeccionamos la estructura de cada DataFrame: dimensiones, columnas, tipos de datos y primeras filas. El objetivo es identificar:

- Qué columnas actúan como identificadores de nodo (`member_id`, `group_id`, `event_id`) y permiten cruzar tablas.
- Qué features numéricas continuas tenemos disponibles (`lat`, `lon`, `num_members`).
- Qué features categóricas requerirán encoding en el feature engineering (`category_name`, `city`, `state`).
- Qué columnas representan aristas y sus pesos (`weight`).

In [3]:
dfs_meta = {
    "meta_events":  dataframes["meta-events"],
    "meta_groups":  dataframes["meta-groups"],
    "meta_members": dataframes["meta-members"],
}

for name, df in dfs_meta.items():
    print(f"{'='*50}")
    print(f"📄 {name}")
    print(f"  Shape      : {df.shape}")
    print(f"  Columnas   : {df.columns.tolist()}")
    print(f"  Tipos      :\n{df.dtypes.to_string()}")
    print(f"\n  Primeras filas:")
    display(df.head(3))

📄 meta_events
  Shape      : (19307, 4)
  Columnas   : ['event_id', 'group_id', 'name', 'time']
  Tipos      :
event_id      str
group_id    int64
name          str
time          str

  Primeras filas:


,event_id,group_id,name,time
0,243930425,26140018,2017 Nashville Walk to End Alzheimers - Octob...,2017-10-14 12:00:00
1,244208851,25604533,Steak Dinner on the Patio,2017-10-15 00:15:00
2,pxlktnywnbfb,25973656,Schedule Meetup,2017-10-03 23:30:00


📄 meta_groups
  Shape      : (602, 7)
  Columnas   : ['group_id', 'group_name', 'num_members', 'category_id', 'category_name', 'organizer_id', 'group_urlname']
  Tipos      :
group_id         int64
group_name         str
num_members      int64
category_id      int64
category_name      str
organizer_id     int64
group_urlname      str

  Primeras filas:


,group_id,group_name,num_members,category_id,category_name,organizer_id,group_urlname
0,339011,Nashville Hiking Meetup,15838,23,Outdoors & Adventure,4353803,nashville-hiking
1,19728145,Stepping Out Social Dance Meetup,1778,5,Dancing,118484462,steppingoutsocialdance
2,6335372,Nashville soccer,2869,32,Sports & Recreation,108448302,Nashville-soccer


📄 meta_members
  Shape      : (24591, 7)
  Columnas   : ['member_id', 'name', 'hometown', 'city', 'state', 'lat', 'lon']
  Tipos      :
member_id      int64
name             str
hometown         str
city             str
state            str
lat          float64
lon          float64

  Primeras filas:


,member_id,name,hometown,city,state,lat,lon
0,2069,Wesley Duffee-Braun,Brentwood,Brentwood,TN,36.00,-86.79
1,8386,Tim,Nashville,Nashville,TN,36.07,-86.78
2,9205,Brenda,Brentwood,Brentwood,TN,36.00,-86.79


## 3. Calidad de Datos

Antes de alimentar cualquier modelo, garantizamos la **integridad** de las tablas base revisando duplicados y valores nulos. El orden importa: siempre eliminamos duplicados primero, ya que las filas clonadas inflan artificialmente las estadísticas de nulos y pueden distorsionar el peso de las aristas en el grafo.

### 3.1 Duplicados

Buscamos filas idénticamente repetidas en todos los DataFrames. La presencia de duplicados en los archivos de aristas sería especialmente problemática, ya que duplicaría el peso de conexiones concretas y alteraría la topología del grafo.

In [4]:
for name, df in dataframes.items():
    total_duplicates = df.duplicated().sum()
    print(f"{name} - Total de filas duplicadas: {total_duplicates}")
    print("-" * 40)

group-edges - Total de filas duplicadas: 0
----------------------------------------
member-edges - Total de filas duplicadas: 0
----------------------------------------
member-to-group-edges - Total de filas duplicadas: 0
----------------------------------------
meta-events - Total de filas duplicadas: 0
----------------------------------------
meta-groups - Total de filas duplicadas: 0
----------------------------------------
meta-members - Total de filas duplicadas: 0
----------------------------------------
rsvps - Total de filas duplicadas: 0
----------------------------------------


Ningún archivo presenta filas duplicadas — el dataset está limpio en este aspecto.

### 3.2 Valores Nulos

Contabilizamos los valores faltantes por archivo. Un nodo con metadatos nulos puede quedar sin representación útil durante el entrenamiento del GAE, por lo que es importante identificar y tratar estos casos antes del feature engineering.

In [5]:
for name, df in dataframes.items():
    total_nulls = df.isna().sum().sum()
    print(f"{name} - Total de valores nulos: {total_nulls}")
    print("-" * 40)

group-edges - Total de valores nulos: 0
----------------------------------------
member-edges - Total de valores nulos: 0
----------------------------------------
member-to-group-edges - Total de valores nulos: 0
----------------------------------------
meta-events - Total de valores nulos: 0
----------------------------------------
meta-groups - Total de valores nulos: 0
----------------------------------------
meta-members - Total de valores nulos: 19758
----------------------------------------
rsvps - Total de valores nulos: 0
----------------------------------------


El único archivo con nulos es `meta-members`. Inspeccionamos el detalle por columna:

In [6]:
print(dataframes["meta-members"].isna().sum())
print("-" * 40)
print((dataframes["meta-members"].isna().sum() / len(dataframes["meta-members"]) * 100).round(2).astype(str) + '%')

member_id        0
name             0
hometown     19664
city             0
state           94
lat              0
lon              0
dtype: int64
----------------------------------------
member_id      0.0%
name           0.0%
hometown     79.96%
city           0.0%
state         0.38%
lat            0.0%
lon            0.0%
dtype: str


Se identifican dos columnas con nulos en `meta-members`:

- **`hometown`** — 19.664 nulos (79.96%). Casi el 80% de los miembros no tiene ciudad natal registrada. Cualquier intento de imputación introduciría un sesgo inaceptable, ya que estaríamos inventando información para 4 de cada 5 nodos. **Decisión: eliminar la columna completa.**

- **`state`** — 94 nulos (0.38%). Al ser menos del 1%, se podría borrar simplemente esa docena de filas concretas o usar técnicas de geolocalización inversa usando las coordenadas `lat`/`lon` que sí están completas al 100%
El análisis cualitativo de estos registros se realiza en el EDA de metadatos, donde se 
determina la estrategia de tratamiento adecuada.

In [7]:
# Eliminamos hometown — 79.96% de nulos hace inviable cualquier imputación
dataframes["meta-members"] = dataframes["meta-members"].drop("hometown", axis=1)

## 4. Exportación de Datos Limpios

Una vez validada la calidad de los datos, guardamos los DataFrames resultantes en `data/processed/`. A partir de este punto, todos los notebooks posteriores (EDA, feature engineering, GAE) se alimentarán exclusivamente de este directorio, garantizando la reproducibilidad del pipeline.

Se guarda con `index=False` para evitar que pandas añada una columna de índice numérico que podría confundirse con un identificador real.

In [8]:
out_dir = "../data/processed"
os.makedirs(out_dir, exist_ok=True)

for name, df in dataframes.items():
    df.to_csv(f"{out_dir}/{name}.csv", index=False)
    filas, columnas = df.shape
    print(f"✅ Guardado: {name}.csv ({filas} filas x {columnas} columnas)")
    print("-" * 50)

✅ Guardado: group-edges.csv (6692 filas x 3 columnas)
--------------------------------------------------
✅ Guardado: member-edges.csv (1176368 filas x 3 columnas)
--------------------------------------------------
✅ Guardado: member-to-group-edges.csv (45583 filas x 3 columnas)
--------------------------------------------------
✅ Guardado: meta-events.csv (19307 filas x 4 columnas)
--------------------------------------------------
✅ Guardado: meta-groups.csv (602 filas x 7 columnas)
--------------------------------------------------
✅ Guardado: meta-members.csv (24591 filas x 6 columnas)
--------------------------------------------------
✅ Guardado: rsvps.csv (126813 filas x 3 columnas)
--------------------------------------------------
